# Hyperliquid Perpetual Futures Data Import

This notebook imports:
1. **OHLC data** (prices and volume) for Hyperliquid perpetual futures
2. **Funding rate data** (hourly funding rates)

## About Hyperliquid
- **Type**: Decentralized perpetual futures exchange
- **Available**: Globally (no geo-restrictions like CEX with BaFin)
- **Contracts**: 229+ perpetual futures
- **Settlement**: USDC (all contracts are linear/USDC-margined)
- **Funding**: Hourly funding rates

## API Details
- **Endpoint**: https://api.hyperliquid.xyz/info
- **Ticker Format**: Simple coin names (BTC, ETH, SOL)
- **Data Range**: Historical data from 2023+
- **Rate Limit**: ~200 req/min

In [ ]:
from clients.hyperliquid_client import HyperliquidClient
from clients.db_client import DBClient
import pandas as pd
from datetime import datetime

In [ ]:
# Configuration
EXCHANGE = 'hyperliquid'
INTERVAL = '1m'  # 1h: 1 hour candles, 1m: 1 minute candles
START_DATE = '2025-01-01' # Only the most recent 5000 candles are available for the OHLCV
END_DATE = datetime.today().strftime('%Y-%m-%d') # Use today's date as the end date

# Test with Bitcoin perpetual
test_symbol = 'BTC'

hl = HyperliquidClient()

## Test Download - Single Contract

In [ ]:


# Test OHLC download
df_ohlc = hl.download_ohlc(
    symbol=test_symbol,
    interval=INTERVAL,
    start_date=START_DATE,
    end_date=END_DATE
)

print(f"\nOHLC Data Preview:")
print(df_ohlc.head())
print(f"\nShape: {df_ohlc.shape}")
print(f"Date range: {df_ohlc['timestamp'].min()} to {df_ohlc['timestamp'].max()}")

In [ ]:
# Debug: Check what the API actually returns for 1m data
import json
from datetime import datetime, timezone, timedelta

test_symbol = 'BTC'
interval = '1m'

# Test with HISTORIC data (2025-01-01)
historic_start = datetime(2025, 1, 1, tzinfo=timezone.utc)
now = datetime.now(timezone.utc)

start_ms = int(historic_start.timestamp() * 1000)
end_ms = int((historic_start + timedelta(days=1)).timestamp() * 1000)

payload = {
    "type": "candleSnapshot",
    "req": {
        "coin": test_symbol,
        "interval": interval,
        "startTime": start_ms,
        "endTime": end_ms
    }
}

print(f"Payload: {json.dumps(payload, indent=2)}")
print(f"\nFetching {test_symbol} {interval} from {datetime.fromtimestamp(start_ms/1000, tz=timezone.utc)} to {datetime.fromtimestamp(end_ms/1000, tz=timezone.utc)}")

response = hl._make_request(payload)

print(f"\nResponse type: {type(response)}")
print(f"Response length: {len(response) if isinstance(response, list) else 'N/A'}")
print(f"Response: {response}")

# Now test with RECENT data (last 2 days)
print("\n" + "="*80)
print("Now testing with RECENT data (last 2 days):")
print("="*80)

recent_end_ms = int(now.timestamp() * 1000)
recent_start_ms = int((now - timedelta(days=2)).timestamp() * 1000)

payload_recent = {
    "type": "candleSnapshot",
    "req": {
        "coin": test_symbol,
        "interval": interval,
        "startTime": recent_start_ms,
        "endTime": recent_end_ms
    }
}

print(f"\nPayload: {json.dumps(payload_recent, indent=2)}")
print(f"\nFetching {test_symbol} {interval} from {datetime.fromtimestamp(recent_start_ms/1000, tz=timezone.utc)} to {datetime.fromtimestamp(recent_end_ms/1000, tz=timezone.utc)}")

response_recent = hl._make_request(payload_recent)

print(f"\nResponse type: {type(response_recent)}")
print(f"Response length: {len(response_recent) if isinstance(response_recent, list) else 'N/A'}")
print(f"Response: {response_recent[:3] if isinstance(response_recent, list) and len(response_recent) > 0 else response_recent}")

In [ ]:
# Test funding rate download
df_funding = hl.download_funding_rates(
    symbol=test_symbol,
    start_date=START_DATE,
    end_date=END_DATE
)

In [ ]:
print(f"Funding Rate Data Preview:")
print(df_funding.head())
print(f"Date range: {df_funding['timestamp'].min()} to {df_funding['timestamp'].max()}")

## Import OHLC data for all Hyperliquid perpetual futures

In [ ]:
with DBClient() as db:
    instruments_dict = db.get_perpetuals_dict(EXCHANGE)
instruments_dict

In [ ]:
with DBClient() as db:
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing {ticker}...")
        
        # Download OHLC data
        df_ohlc = hl.download_ohlc(
            symbol=ticker,
            interval=INTERVAL,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_ohlc.empty:
            print(f"No OHLC data available for {ticker}")
            continue
        
        # Check existing data range
        min_ts, max_ts = db.get_timestamp_range(instrument_id)
        
        # Check for duplicates
        if min_ts and max_ts:
            # Filter out overlapping data
            df_ohlc = df_ohlc[(df_ohlc['timestamp'] < min_ts) | (df_ohlc['timestamp'] > max_ts)]
                
            if df_ohlc.empty:
                print(f"All data already exists, skipping")
                continue

        # Add instrument_id and insert
        df_ohlc['instrument_id'] = instrument_id
        rows_inserted = db.insert_market_data(df_ohlc)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} OHLC records")
        else:
            print(f"No new OHLC records inserted (duplicates?)")

    print("OHLC IMPORT COMPLETED")

## Export OHLC data to Parquet (no DB import)

In [ ]:
import os
from pathlib import Path

# Create data directory if it doesn't exist
data_dir = Path('../../data')
data_dir.mkdir(parents=True, exist_ok=True)

all_ohlc_data = []

for ticker, instrument_id in instruments_dict.items():
    print(f"Downloading {ticker}...")
    
    # Download OHLC data
    df_ohlc = hl.download_ohlc(
        symbol=ticker,
        interval=INTERVAL,
        start_date=START_DATE,
        end_date=END_DATE
    )
    
    if df_ohlc.empty:
        print(f"No OHLC data for {ticker}")
        continue
    
    # Add ticker column for easy filtering
    df_ohlc['ticker'] = ticker
    all_ohlc_data.append(df_ohlc)

if all_ohlc_data:
    # Combine all data
    df_all = pd.concat(all_ohlc_data, ignore_index=True)
    
    # Save to parquet with compression
    output_path = data_dir / f'hyperliquid_ohlc_{INTERVAL}_{START_DATE}_to_{END_DATE}.parquet'
    df_all.to_parquet(output_path, compression='snappy', index=False)
    
    print(f"\nSaved {len(df_all):,} records to {output_path}")
    print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("No data collected")

## Import funding rate data for all Hyperliquid perpetual futures

In [ ]:
with DBClient() as db:
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing funding for {ticker}...")
        
        # Download funding rate data
        df_funding = hl.download_funding_rates(
            symbol=ticker,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_funding.empty:
            print(f"No funding data available for {ticker}")
            continue
        
        # Check existing funding data range
        min_ts, max_ts = db.get_funding_timestamp_range(instrument_id)
        
        # Check for duplicates
        if min_ts and max_ts:
            # Filter out overlapping data
            df_funding = df_funding[(df_funding['timestamp'] < min_ts) | (df_funding['timestamp'] > max_ts)]
                
            if df_funding.empty:
                print(f"All funding data already exists for {ticker}, skipping")
                continue

        # Add instrument_id and insert
        df_funding['instrument_id'] = instrument_id
        rows_inserted = db.insert_funding_data(df_funding)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} funding records for {ticker}")
        else:
            print(f"No new funding records inserted for {ticker} (duplicates?)")

    print("FUNDING RATE IMPORT COMPLETED")